In [1]:
#!pip install tqdm

In [2]:
import numpy as np
from scipy import special
import scipy
import random
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp  # solve_ivp plus robuste que odeint et permet de remedier aux excessive work
import seaborn as sns
from matplotlib import cm
from matplotlib.ticker import LinearLocator
import pandas as pd
from SALib.sample import fast_sampler  # echantillonage pour efast, LHS fait aussi bien l'affaire
from SALib.analyze import fast  
from SALib.plotting.bar import plot as barplot  # eFAST visualization
from tqdm import tqdm

In [3]:
#COnfig 
#style.use('seaborn')
plt.rcParams['figure.dpi'] = 1200
plt.rcParams['axes.labelsize'] = 8
plt.rcParams['axes.titlesize'] = 8
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['lines.linewidth'] = 1

In [4]:
# Parameter baselines - # 6 parametres au total + N: pop totle
Lambdab = 5000
betab = 0.5
gammab = 0.1
alphab = 0.09
mub = 0.0019 
deltab = 0.09
#NTb = 8500

In [5]:
# Define the problem for eFAST
problem = {
    'num_vars': 6,  # Numbre de params
    'names': ['Lambda', 'beta', 'gamma', 'alpha', 'mu', 'delta' ],  
    'bounds': [
        [Lambdab * 0.7, Lambdab * 1.3],  # Lambda
        [betab * 0.7, betab * 1.3],  # beta
        [gammab * 0.7, gammab * 1.3],  # gamma
        [alphab * 0.7, alphab * 1.3],  # alpha
        [mub * 0.7, mub * 1.3],  # mu
        [deltab * 0.7, deltab * 1.3],  # delta
    ]  
}

In [6]:
N = 66  # Number of samplingg doit etre superieur a 65 (4 M ^ 2 )
param_values = fast_sampler.sample(problem, N)

# data frame
headerList = problem['names']
df = pd.DataFrame(param_values, columns=headerList)
df.to_csv("parameter_combination_eFAST.csv", index=False)
df = pd.read_csv("parameter_combination_eFAST.csv")

In [7]:
# Define the ODE model
def myodes(t, y, Lambda, beta, gamma, alpha, mu, delta ):
    S, I, R = y
    
    dS = Lambda - beta * S * I + alpha * I - mu * S
    dI = beta * S * I - (alpha + gamma + mu + delta) * I
    dR = gamma * I - mu * R

    return [dS, dI, dR]

In [8]:
# Output computations
outputs = {'S': [], 'I': [], 'R': [] }
t_span = (0, 100) 
t_eval = np.linspace(0, 100, 100)  

In [9]:
for i in range(len(df)):
    sampledParams = df.iloc[i].values
    y0 = [5, 2, 1]  # Initial conditions
    
    # Solve the ODE using solve_ivp
    sol = solve_ivp(myodes, t_span, y0, args=tuple(sampledParams), t_eval=t_eval, method='LSODA')
    
    outputs['S'].append(sol.y[0][-1])  #  S
    outputs['I'].append(sol.y[1][-1])  #  I
    outputs['R'].append(sol.y[2][-1])  #  R

In [10]:
for key in outputs:
    df[key] = outputs[key]

In [11]:
latex_names = [
   r'$\Lambda$', r'$\beta$', r'$\gamma$', r'$\alpha$', r'$\mu$', r'$\delta$'
]

j = 0
for output in ['S', 'I', 'R']:
    Y = df[output].values
    Si = fast.analyze(problem, Y)
    
    # Convert ResultDict to DataFrame
    Si_df = pd.DataFrame({
        'Parameter': problem['names'],
        'S1': Si['S1'],
        'ST': Si['ST']
    })
    outp = ['$S$', '$I$', '$R$']

    
    # Plot sensitivity indices
    fig, ax = plt.subplots(figsize=(10, 5))
    #barplot(Si_df, ax=ax)  # Pass the DataFrame to barplot
    ax.bar(latex_names, Si_df['ST'], color = 'darkorange', label = 'ST') 
    ax.bar(latex_names, Si_df['S1'], color = 'dodgerblue', label = 'S1') # Pass the DataFrame to barplot
    ax.set_title(f"eFAST Sensitivity Indices for {outp[j]}")
    
    
    
    j = j + 1
    # Set LaTeX labels for x-axis
    ax.legend() 
    ax.set_xticks(range(len(latex_names)))  # Positions des ticks
    ax.set_xticklabels(latex_names, rotation=90, fontsize=8)  # Labels en LaTeX
    
    plt.tight_layout()
    plt.savefig(f"eFAST_{output}.pdf", bbox_inches='tight')       
    plt.show()

   